In [1]:
import torch
import torch.nn as nn

# 模拟参数
in_features, out_features, r = 512, 512, 8 # 低秩 r=8
W0 = torch.randn(in_features, out_features) # 模拟预训练权重

# --- 测试 1: 普通 LoRA (对比组) ---
A_lora = nn.Parameter(torch.randn(in_features, r))
B_lora = nn.Parameter(torch.randn(r, out_features))
delta_W_lora = A_lora @ B_lora # 矩阵乘法
rank_lora = torch.linalg.matrix_rank(delta_W_lora).item()
print(f"LoRA 更新矩阵的秩: {rank_lora}") # 理论上 <= r (通常为 8)

# --- 测试 2: Hi-DoRA (实验组) ---
# 使用上面生成的 A, B 和 W0
AB = A_lora @ B_lora
hira_direction_delta = W0 * AB # 这里的 W0 是上面的模拟权重
# 注意：在 Hi-DoRA 中，实际更新是 m * (V_new / Norm) - 原始方向
# 但为了看更新量的秩，我们看方向的变化量
rank_hidora = torch.linalg.matrix_rank(hira_direction_delta).item()
print(f"Hi-DoRA 方向增量的秩: {rank_hidora}") # 理论上远大于 r (可能接近 512)

LoRA 更新矩阵的秩: 8
Hi-DoRA 方向增量的秩: 512


In [11]:
# 检查范数
import sys
sys.path.append("..")

import math
from .aeloru_layer import HiDoRALayer
hidora_layer = HiDoRALayer(in_features, out_features, r=8)
W_prime = hidora_layer(W0)

# 计算输出权重的范数
output_norm = torch.norm(W_prime, dim=0) # 按列计算
print(f"输出权重的范数分布: Mean={output_norm.mean().item():.4f}, Std={output_norm.std().item():.4f}")

# 对比 DoRA 的理想情况：范数应主要由 self.m 控制
# 如果 Std 很小，说明方向被很好地归一化了

ImportError: attempted relative import with no known parent package

# prat 2

In [7]:

# 导入必要库
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import Dataset
import numpy as np
import evaluate
from tqdm import tqdm

e:\Conda\envs\Qelys\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import sys
sys.path.append('..')  # 添加父目录到路径
from data.gsm8k.gsm8k import Gsm8k

e:\Conda\envs\Qelys\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

# 加载 GSM8K 数据集
ds = Gsm8k._split_generators

def load_dataset(ds):
    data = []
    "遍历数据集"
    for split in ds:
        for example in split.gen_kwargs['data_files']['train']:
            data.append(example)
    return data
ds = load_dataset(ds)
# 数据预处理：将问题和答案组合成适合微调的格式
def preprocess_function(examples):
    inputs = [f"问题: {question}\n答案: " for question in examples["question"]]
    targets = examples["answer"]
    return {"input": inputs, "target": targets}



TypeError: 'function' object is not iterable

In [ ]:
import torch.nn as nn
from peft.tuners.lora import LoraLayer

class HiDoRALayer(LoraLayer):
    """Hi-DoRA 实现：结合 HiRA 高秩特性和 DoRA 幅度/方向解耦"""
    
    def __init__(self, in_features, out_features, r=8):
        super().__init__(in_features, out_features, r)
        # 初始化 LoRA 参数
        self.lora_A = nn.Parameter(torch.randn(in_features, r))
        self.lora_B = nn.Parameter(torch.randn(r, out_features))
        # 初始化 DoRA 幅度参数
        self.m = nn.Parameter(torch.ones(out_features))
    
    def forward(self, x, *args, **kwargs):
        # 原始权重 W0
        W0 = self.weight
        # LoRA 更新 AB
        AB = self.lora_A @ self.lora_B
        # HiRA 方向增量：W0 ⊙ AB
        hira_direction_delta = W0 * AB
        # DoRA 归一化：m * (V_new / Norm)
        V_new = W0 + hira_direction_delta
        norm = torch.norm(V_new, dim=0)
        W_prime = self.m * (V_new / norm)
        # 应用更新后的权重
        self.weight = W_prime
        return super().forward(x, *args, **kwargs)

In [ ]:
# 加载 Qwen 基础模型
model_name = "models/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 设置实验对比组
experiments = {
    "LoRA": {
        "config": LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"]),
        "description": "标准 LoRA 微调"
    },
    "Hi-DoRA": {
        "config": LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"], 
                              peft_type="HIDORA", hidora_layer=HiDoRALayer),
        "description": "结合 HiRA 高秩特性和 DoRA 幅度/方向解耦"
    }
}

# 应用 PEFT 配置
for name, exp in experiments.items():
    model = get_peft_model(model, exp["config"])
    experiments[name]["model"] = model

'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/models/Qwen2.5-0.5B/resolve/main/config.json
Retrying in 1s [Retry 1/5].


KeyboardInterrupt: 

In [ ]:
def train_and_evaluate(experiment_name, model, dataset, epochs=3):
    """训练并评估模型在 GSM8K 上的性能"""
    
    # 准备训练数据
    train_dataset = dataset.shuffle().select(range(1000))  # 使用 1000 个样本进行训练
    
    # 训练参数
    training_args = {
        "learning_rate": 5e-5,
        "per_device_train_batch_size": 4,
        "num_train_epochs": epochs,
        "logging_steps": 100,
        "save_steps": 500,
        "evaluation_strategy": "no",
        "report_to": "none"
    }
    
    # 训练过程
    print(f"开始训练 {experiment_name}...")
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=training_args["learning_rate"])
    
    for epoch in range(epochs):
        for i in tqdm(range(0, len(train_dataset), training_args["per_device_train_batch_size"])):
            batch = train_dataset[i:i+training_args["per_device_train_batch_size"]]
            inputs = tokenizer(batch["input"], padding=True, truncation=True, return_tensors="pt")
            labels = tokenizer(batch["target"], padding=True, truncation=True, return_tensors="pt")["input_ids"]
            
            # 前向传播
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
    
    # 评估过程
    print(f"评估 {experiment_name}...")
    model.eval()
    eval_dataset = dataset.select(range(1000, 1500))  # 使用 500 个样本进行评估
    metric = evaluate.load("accuracy")
    
    for i in tqdm(range(0, len(eval_dataset), training_args["per_device_train_batch_size"])):
        batch = eval_dataset[i:i+training_args["per_device_train_batch_size"]]
        inputs = tokenizer(batch["input"], padding=True, truncation=True, return_tensors="pt")
        
        # 生成预测
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=512)
        
        # 解码预测和真实答案
        predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        references = batch["target"]
        
        # 计算准确率
        metric.add_batch(predictions=predictions, references=references)
    
    results = metric.compute()
    print(f"{experiment_name} 准确率: {results['accuracy']:.4f}")
    return results["accuracy"]

In [ ]:
# 运行对比实验
results = {}
for name, exp in experiments.items():
    accuracy = train_and_evaluate(name, exp["model"], ds)
    results[name] = accuracy

# 输出最终结果对比
print("\n实验结果对比:")
for name, accuracy in results.items():
    print(f"{name}: {accuracy:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# 绘制 Loss 曲线对比
plt.figure(figsize=(10, 6))
for name, exp in experiments.items():
    plt.plot(exp["loss_history"], label=name)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LoRA vs Hi-DoRA Loss 收敛对比")
plt.legend()
plt.show()

In [ ]:
# 消融实验：仅 HiRA
experiments["Only_HiRA"] = {
    "config": LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"],
                          peft_type="HIDORA", hidora_layer=HiDoRALayer, 
                          disable_dora=True),  # 禁用 DoRA 归一化
    "description": "仅使用 HiRA 高秩特性"
}

# 消融实验：仅 DoRA
experiments["Only_DoRA"] = {
    "config": LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"],
                          peft_type="DORA"),
    "description": "仅使用 DoRA 幅度/方向解耦"
}

In [3]:
import numpy as np

a = np.array(
    [[2,2],
     [3,4]]
)
for i in range(100+1):
    a *=np.array([[2,2],[3,4]])
    if i % 10 == 0:
        print(f"time {i}: {a}")
    

time 0: [[ 4  4]
 [ 9 16]]
time 10: [[    4096     4096]
 [  531441 16777216]]
time 20: [[       4194304        4194304]
 [   31381059609 17592186044416]]
time 30: [[      4294967296       4294967296]
 [1853020188851841                0]]
time 40: [[       4398046511104        4398046511104]
 [-1261475310744950487                    0]]
time 50: [[   4503599627370496    4503599627370496]
 [-903054539411881455                   0]]
time 60: [[4611686018427387904 4611686018427387904]
 [5069619362125685561                   0]]
time 70: [[                  0                   0]
 [2190886001003067041                   0]]
time 80: [[                  0                   0]
 [2611284305020221001                   0]]
time 90: [[                   0                    0]
 [-2606784999112070095                    0]]
time 100: [[                   0                    0]
 [-8414861536128355751                    0]]


In [1]:
import numpy as np

a = np.array(
    [[0,1],
     [1,1]]
)
for i in range(11):
    a *=np.array([[0,1],[1,1]])
    print(f"time {i}: {a}")

time 0: [[0 1]
 [1 1]]
time 1: [[0 1]
 [1 1]]
time 2: [[0 1]
 [1 1]]
time 3: [[0 1]
 [1 1]]
time 4: [[0 1]
 [1 1]]
time 5: [[0 1]
 [1 1]]
time 6: [[0 1]
 [1 1]]
time 7: [[0 1]
 [1 1]]
time 8: [[0 1]
 [1 1]]
time 9: [[0 1]
 [1 1]]
time 10: [[0 1]
 [1 1]]
